# Embedding Features Redundancy Analysis

这个 notebook 用于分析当前 embedding 管线中的上下文特征是否存在冗余，重点检查：

1. 常量/低变化特征
2. 数值桶特征间的高相关
3. 类别特征间的近确定映射（一个特征几乎能唯一决定另一个）
4. 各特征对标签（next city token）的信息量（Mutual Information，可选）

> 数据与特征生成逻辑直接复用项目代码，保证与训练脚本一致。

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Ensure project root is in sys.path
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    ROOT = cwd
else:
    ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), cwd)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import (
    create_multiple_sequences,
    build_city_vocab,
    build_booker_device_vocabs,
    build_hotel_country_vocab,
    build_city_sequence_pack,
)

DATA_DIR = ROOT / "data"
train_set = pd.read_csv(DATA_DIR / "train_set.csv")

print(f"ROOT={ROOT}")
print(f"train_set shape={train_set.shape}")

ROOT=/root/gr/Generative-Retrieval-for-Multi-Destination-Trips
train_set shape=(1166835, 9)


In [2]:
# Build the exact same training pack as embedding pipeline
train_trips = create_multiple_sequences(train_set)
city_to_idx, _ = build_city_vocab(train_set)
booker_to_idx, device_to_idx, _, _ = build_booker_device_vocabs(train_trips)
hotel_country_to_idx, _ = build_hotel_country_vocab(train_trips)

# multi_step=True gives richer sample coverage for feature analysis
train_pack = build_city_sequence_pack(
    train_trips,
    city_to_idx,
    is_test=False,
    multi_step=True,
    booker_to_idx=booker_to_idx,
    device_to_idx=device_to_idx,
    hotel_country_to_idx=hotel_country_to_idx,
)

feature_df = pd.DataFrame(
    {
        "ctx_booker": train_pack.ctx_booker,
        "ctx_device": train_pack.ctx_device,
        "ctx_month": train_pack.ctx_month,
        "ctx_stay": train_pack.ctx_stay,
        "ctx_trip_len": train_pack.ctx_trip_len,
        "ctx_num_unique_cities": train_pack.ctx_num_unique_cities,
        "ctx_repeat_city_ratio": train_pack.ctx_repeat_city_ratio,
        "ctx_last_stay_days": train_pack.ctx_last_stay_days,
        "ctx_same_country_streak": train_pack.ctx_same_country_streak,
        "ctx_last_hotel_country": train_pack.ctx_last_hotel_country,
        "ctx_unique_hotel_countries": train_pack.ctx_unique_hotel_countries,
        "ctx_cross_border_count": train_pack.ctx_cross_border_count,
        "ctx_cross_border_ratio": train_pack.ctx_cross_border_ratio,
    }
)
feature_df["target_next_city_token"] = train_pack.y

print(f"samples={len(feature_df):,}")
feature_df.head()

samples=949,149


,ctx_booker,ctx_device,ctx_month,ctx_stay,ctx_trip_len,ctx_num_unique_cities,ctx_repeat_city_ratio,ctx_last_stay_days,ctx_same_country_streak,ctx_last_hotel_country,ctx_unique_hotel_countries,ctx_cross_border_count,ctx_cross_border_ratio,target_next_city_token
0,2,1,8,1,1,1,1,1,1,61,1,0,0,9212
1,2,1,8,2,2,2,1,2,2,61,1,0,0,36013
2,2,1,8,2,3,3,1,2,3,61,1,0,0,18090
3,3,2,4,2,1,1,1,2,1,37,1,0,0,30816
4,3,2,4,2,2,2,1,1,2,37,1,0,0,12599


In [3]:
# 1) Basic sanity: constants / low-variance / cardinality
summary = pd.DataFrame(
    {
        "dtype": feature_df.dtypes.astype(str),
        "n_unique": feature_df.nunique(dropna=False),
        "missing_ratio": feature_df.isna().mean(),
    }
).sort_values("n_unique")

print("Constant columns:", summary.index[summary["n_unique"] <= 1].tolist())
summary

Constant columns: []


,dtype,n_unique,missing_ratio
ctx_device,int64,3,0.0
ctx_booker,int64,5,0.0
ctx_repeat_city_ratio,int64,10,0.0
ctx_cross_border_ratio,int64,11,0.0
ctx_month,int64,12,0.0
ctx_unique_hotel_countries,int64,19,0.0
ctx_cross_border_count,int64,21,0.0
ctx_stay,int64,28,0.0
ctx_last_stay_days,int64,29,0.0
ctx_num_unique_cities,int64,30,0.0


In [4]:
# 2) Correlation analysis for bucket-like numeric features
numeric_cols = [
    "ctx_stay",
    "ctx_trip_len",
    "ctx_num_unique_cities",
    "ctx_repeat_city_ratio",
    "ctx_last_stay_days",
    "ctx_same_country_streak",
    "ctx_unique_hotel_countries",
    "ctx_cross_border_count",
    "ctx_cross_border_ratio",
]

corr = feature_df[numeric_cols].corr(method="spearman")
high_corr_pairs = []
threshold = 0.90
for i, c1 in enumerate(numeric_cols):
    for c2 in numeric_cols[i + 1 :]:
        v = corr.loc[c1, c2]
        if abs(v) >= threshold:
            high_corr_pairs.append((c1, c2, float(v)))

print(f"High-correlation pairs (|rho| >= {threshold}):")
for p in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(p)

corr

High-correlation pairs (|rho| >= 0.9):
('ctx_unique_hotel_countries', 'ctx_cross_border_count', 0.9990615291799484)
('ctx_unique_hotel_countries', 'ctx_cross_border_ratio', 0.9931031066645493)
('ctx_cross_border_count', 'ctx_cross_border_ratio', 0.9926295662494147)
('ctx_trip_len', 'ctx_num_unique_cities', 0.9397735410109549)


,ctx_stay,ctx_trip_len,ctx_num_unique_cities,ctx_repeat_city_ratio,ctx_last_stay_days,ctx_same_country_streak,ctx_unique_hotel_countries,ctx_cross_border_count,ctx_cross_border_ratio
ctx_stay,1.000000,-0.055398,-0.038177,-0.038461,0.742857,-0.074569,0.058874,0.058258,0.065567
ctx_trip_len,-0.055398,1.000000,0.939774,0.263976,-0.066319,0.827444,0.292559,0.295137,0.258602
ctx_num_unique_cities,-0.038177,0.939774,1.000000,-0.024879,-0.046486,0.762493,0.310386,0.312557,0.279277
ctx_repeat_city_ratio,-0.038461,0.263976,-0.024879,1.000000,-0.056149,0.256604,-0.002353,-0.001023,-0.012444
ctx_last_stay_days,0.742857,-0.066319,-0.046486,-0.056149,1.000000,-0.090843,0.059757,0.058318,0.065625
ctx_same_country_streak,-0.074569,0.827444,0.762493,0.256604,-0.090843,1.000000,-0.204042,-0.204385,-0.227243
ctx_unique_hotel_countries,0.058874,0.292559,0.310386,-0.002353,0.059757,-0.204042,1.000000,0.999062,0.993103
ctx_cross_border_count,0.058258,0.295137,0.312557,-0.001023,0.058318,-0.204385,0.999062,1.000000,0.992630
ctx_cross_border_ratio,0.065567,0.258602,0.279277,-0.012444,0.065625,-0.227243,0.993103,0.992630,1.000000


In [5]:
# 3) Near-deterministic mapping detection
# "a -> b" score: for each value of a, how often b is almost uniquely determined.
def directional_determinism(df: pd.DataFrame, a: str, b: str) -> float:
    gb = df.groupby(a)[b].nunique(dropna=False)
    # fraction of groups where b is unique
    return float((gb <= 1).mean())

all_feature_cols = [c for c in feature_df.columns if c != "target_next_city_token"]
det_rows = []
for i, a in enumerate(all_feature_cols):
    for b in all_feature_cols:
        if a == b:
            continue
        score = directional_determinism(feature_df, a, b)
        if score >= 0.98:  # strong sign of redundancy / leakage-like encoding relation
            det_rows.append((a, b, score))

det_df = pd.DataFrame(det_rows, columns=["source", "target", "determinism"]).sort_values(
    "determinism", ascending=False
)
print("Potential near-deterministic relations (source -> target):")
det_df.head(30)

Potential near-deterministic relations (source -> target):


,source,target,determinism


In [6]:
# 4) Mutual Information with target (optional but useful)
# If sklearn is unavailable, this cell will skip gracefully.
try:
    from sklearn.feature_selection import mutual_info_classif

    X = feature_df.drop(columns=["target_next_city_token"])
    y = feature_df["target_next_city_token"]

    # All features are discrete/bucketed IDs
    mi = mutual_info_classif(X, y, discrete_features=True, random_state=42)
    mi_df = pd.DataFrame({"feature": X.columns, "mutual_info": mi}).sort_values(
        "mutual_info", ascending=False
    )
    mi_df
except Exception as e:
    print("Skip MI analysis:", repr(e))

In [7]:
# 5) Auto-generated candidate recommendations
recommendations = []

# From high correlation
for c1, c2, rho in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    recommendations.append(
        f"High correlation: {c1} vs {c2} (spearman={rho:.3f}) -> consider dropping one or using regularization"
    )

# From near-deterministic mapping
if not det_df.empty:
    for _, r in det_df.head(10).iterrows():
        recommendations.append(
            f"Near-deterministic mapping: {r['source']} -> {r['target']} (score={r['determinism']:.3f})"
        )

# From MI (if available)
if "mi_df" in globals():
    low_mi = mi_df[mi_df["mutual_info"] < mi_df["mutual_info"].quantile(0.2)]["feature"].tolist()
    if low_mi:
        recommendations.append(
            "Low-MI features (bottom 20%): " + ", ".join(low_mi)
        )

if recommendations:
    print("Potential actions:")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
else:
    print("No obvious severe redundancy found under current thresholds.")

Potential actions:
1. High correlation: ctx_unique_hotel_countries vs ctx_cross_border_count (spearman=0.999) -> consider dropping one or using regularization
2. High correlation: ctx_unique_hotel_countries vs ctx_cross_border_ratio (spearman=0.993) -> consider dropping one or using regularization
3. High correlation: ctx_cross_border_count vs ctx_cross_border_ratio (spearman=0.993) -> consider dropping one or using regularization
4. High correlation: ctx_trip_len vs ctx_num_unique_cities (spearman=0.940) -> consider dropping one or using regularization
5. Low-MI features (bottom 20%): ctx_unique_hotel_countries, ctx_repeat_city_ratio, ctx_device


## How To Use This Notebook

- 先从头执行到尾，得到 `high_corr_pairs`、`det_df`、`mi_df`。
- 若某两个特征高度相关或存在近确定映射，建议先做小规模 ablation：每次删 1 个特征重训 1-2 epoch 看 `Acc@4`。
- 对 MI 很低且业务解释弱的特征，优先尝试移除。
- 最后保留“信息量高 + 稳定增益 + 可解释”的特征集合。